In [ ]:
# @title Run this first — works on Google Colab and locally { display-mode: "form" }
# On Colab: clones the repo and installs packages (~10 min first run).
# Locally:  just runs the smoke test (assumes you already ran bash setup_env.sh).
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # TODO: set this to your public GitHub repo URL before sharing with students.
    REPO = "https://github.com/NeoNeuron/ccnss26-tutorials"
    REPO_DIR = "/content/ccnss26-tutorials"

    if not os.path.exists(REPO_DIR):
        print("Cloning repo…")
        ret = os.system(f"git clone --quiet {REPO} {REPO_DIR}")
        if ret != 0:
            raise RuntimeError(f"git clone failed — check that REPO is set: {REPO!r}")
    sys.path.insert(0, REPO_DIR)

    def _pip(*args):
        cmd = f"{sys.executable} -m pip install -q " + " ".join(args)
        if os.system(cmd) != 0:
            raise RuntimeError(f"pip failed: {cmd}")

    try:
        # fast path: packages already present from a previous run in this session
        import ssm  # noqa
        from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache  # noqa
    except ImportError:
        print("Installing packages (first run takes ~10 min)…")
        # Cython + setuptools must be present before ssm builds its C extensions.
        # We do NOT re-pin numpy here so Colab's already-loaded numpy stays consistent.
        _pip("Cython", "setuptools")
        _pip("--no-build-isolation",
             "'ssm @ git+https://github.com/lindermanlab/ssm.git@master'")
        _pip("pandas", "scikit-learn", "networkx", "pynwb", "dandi")
        # allensdk declares numpy<1.24 and pandas==1.5.3 — bypass with --no-deps,
        # then add its actual runtime requirements individually.
        _pip("allensdk", "--no-deps")
        _pip("SimpleITK", "'xarray<2023.2.0'", "simplejson", "nest-asyncio",
             "psycopg2-binary", "pynrrd", "future", "requests-toolbelt",
             "scikit-image", "statsmodels", "seaborn", "'marshmallow<4.0.0'",
             "argschema", "boto3", "semver", "cachetools", "sqlalchemy")
        _pip("nlb-tools", "--no-deps")
        _pip("-e", f"{REPO_DIR}/neural-data-analysis/")
        print("Packages installed.")

# ── smoke test ────────────────────────────────────────────────────────────────
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache  # noqa
import ssm  # noqa
print("✅ Setup complete.")

# CCNSS 2026 — Session 1: Coding and Networks

**Theme:** From single-neuron coding to network interactions.

We will analyze one Allen Visual Coding Neuropixels session in three modules:
- **1A** — Tuning curves and PSTHs
- **1B** — Signal and noise correlations
- **1C** — Functional networks and graph theory

**Time budget:** 45 minutes. Each module: ~5 min intro → 5–8 min exercise → 2 min reveal.

In [2]:
# @title Setup (run once) { display-mode: "form" }
import sys
# sys.path.insert(0, "/content/ccnss2026-neural-data-analysis")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ccnss_helpers import data, plotting, save_checkpoint, load_or_compute

print("✅ Setup complete.")

✅ Setup complete.


In [ ]:
# Loads ~2 GB Allen NWB on first run (~3-5 min). Subsequent cells re-use the cache.
session = data.load_allen_session(session_id=715093703, cache_dir='~/.ccnss_cache/allen_cache/')
print(f"Session {session['session_id']}: {len(session['units'])} good units across "
      f"{session['units']['ecephys_structure_acronym'].nunique()} areas")
session["units"].groupby("ecephys_structure_acronym").size()

Using AllenSDK cache dir: /Users/kchen/.ccnss_cache/allen_cache


In [ ]:
# Visualize all simultaneously-recorded units for 5 s during drifting gratings,
# coloring each unit by the brain area it was recorded from. Units are sorted by
# area so same-area units form contiguous colored blocks in the raster.
areas = session["units"]["ecephys_structure_acronym"].fillna("unknown")
sample_uids = areas.sort_values().index
sample_spikes = {uid: session["spike_times"][uid] for uid in sample_uids}

unique_areas = list(dict.fromkeys(areas[sample_uids]))  # in plotted (sorted) order
palette = plt.cm.tab20(np.linspace(0, 1, len(unique_areas)))
area_to_color = dict(zip(unique_areas, palette))
unit_colors = {uid: area_to_color[areas[uid]] for uid in sample_uids}

t0 = session["stim_table"]["start_time"].iloc[0]
fig, ax = plt.subplots(figsize=(8, 8))
plotting.plot_raster(sample_spikes, t_start=t0, t_end=t0 + 2, colors=unit_colors, ax=ax)
ax.set_yticks([])
ax.spines[["top", "right"]].set_visible(False)

handles = [plt.Line2D([0], [0], color=area_to_color[a], lw=3) for a in unique_areas]
ax.legend(handles, unique_areas, title="Area", loc="center left",
          bbox_to_anchor=(1.01, 0.5), fontsize=14, frameon=False)
ax.set_title(f"{len(sample_uids)} simultaneously-recorded units by area, "
             "2 s during drifting gratings");


## Module 1A — Tuning curves & PSTHs

**Goal:** for each unit, compute how its firing rate depends on grating orientation.
Then find the most orientation-selective neuron in the population.

**Steps:**
1. Bin spikes into a (n_units × n_trials × n_bins) array aligned to stimulus onset.
2. For each unit, compute mean rate per orientation → tuning curve.
3. Compute the orientation selectivity index (OSI):

$$\mathrm{OSI} = \frac{R_{\text{pref}} - R_{\text{orth}}}{R_{\text{pref}} + R_{\text{orth}}}$$

In [ ]:
def bin_spikes_per_trial(spike_times_dict, stim_table, bin_size_s=0.025, window_s=(0.0, 2.0)):
    """Return (n_units, n_trials, n_bins) spike counts aligned to trial start_time."""
    uids = list(spike_times_dict.keys())
    bin_edges = np.arange(window_s[0], window_s[1] + bin_size_s, bin_size_s)
    n_bins = len(bin_edges) - 1
    n_trials = len(stim_table)
    counts = np.zeros((len(uids), n_trials, n_bins), dtype=np.int32)
    for i, uid in enumerate(uids):
        st = spike_times_dict[uid]
        for j, t0 in enumerate(stim_table["start_time"].to_numpy()):
            rel = st - t0
            mask = (rel >= window_s[0]) & (rel < window_s[1])
            counts[i, j, :] = np.histogram(rel[mask], bins=bin_edges)[0]
    return counts, bin_edges, uids

counts, bin_edges, uids = bin_spikes_per_trial(
    session["spike_times"], session["stim_table"]
)
print(counts.shape)  # expected: (n_units, n_trials, n_bins)

In [ ]:
def compute_tuning_curves(counts, stim_table, bin_size_s=0.025):
    """Mean firing rate per (unit, orientation). Returns (orientations, rates[n_units, n_oris])."""
    rates_per_trial = counts.sum(axis=2) / (counts.shape[2] * bin_size_s)  # Hz
    oris = np.sort(stim_table["orientation"].dropna().unique())
    tc = np.zeros((counts.shape[0], len(oris)))
    for k, ori in enumerate(oris):
        mask = stim_table["orientation"].to_numpy() == ori
        tc[:, k] = rates_per_trial[:, mask].mean(axis=1)
    return oris, tc

oris, tc = compute_tuning_curves(counts, session["stim_table"])
print(tc.shape)

In [ ]:
# EXERCISE: compute OSI for every unit and find the most selective one
# YOUR CODE HERE
raise NotImplementedError('compute OSI for every unit and find the most selective one')


In [ ]:
# EXERCISE: (challenge) two-component Gaussian mixture tuning fit
# YOUR CODE HERE
raise NotImplementedError('(challenge) two-component Gaussian mixture tuning fit')


In [ ]:
# Orientation selectivity index (OSI) for this neuron, unit uids[best].
# Orientation is 180-deg periodic, so the null response is the *orthogonal*
# orientation (90 deg away) -- NOT the opposite drift direction (180 deg away,
# which is the SAME orientation).
tc_row = tc[best]
d_ori = np.diff(oris).min()                       # spacing between sampled directions (deg)
pref_idx = tc_row.argmax()
orth_idx = (pref_idx + int(round(90 / d_ori))) % len(tc_row)
R_pref, R_orth = tc_row[pref_idx], tc_row[orth_idx]
osi_pref_orth = (R_pref - R_orth) / (R_pref + R_orth + 1e-9)

# Global (vector-average) OSI over the full tuning curve; the 2*theta makes it
# 180-deg periodic, and it uses every orientation so it is less noise-sensitive.
theta = np.deg2rad(oris)
osi_global = np.abs(np.sum(tc_row * np.exp(2j * theta))) / (np.sum(tc_row) + 1e-9)

area = session["units"].loc[uids[best], "ecephys_structure_acronym"]
print(f"Unit {uids[best]} ({area}):")
print(f"  pref dir = {oris[pref_idx]} deg, orthogonal dir = {oris[orth_idx]} deg")
print(f"  OSI (pref vs orthogonal) = {osi_pref_orth:.2f}")
print(f"  OSI (global / 1 - circular variance) = {osi_global:.2f}")

In [ ]:
save_checkpoint(
    "module_1A",
    counts=counts,
    bin_edges=bin_edges,
    uids=np.asarray(uids),
    oris=oris,
    tc=tc,
    osis=osis,
)
print("✅ 1A checkpoint saved.")

## Module 1B — Signal and noise correlations

For each pair of neurons (i, j):
- **Signal correlation:** corr of their tuning curves across orientations.
- **Noise correlation:** corr of their trial-to-trial responses at a fixed orientation, averaged across orientations.

Recreate the canonical Cohen & Kohn 2011 scatter plot: r_signal on x, r_noise on y.

In [ ]:
ck = load_or_compute(
    "module_1A",
    fn=lambda: {"error": np.array([1])},  # If we get here in 1B, 1A must have run.
)
counts, oris, tc, uids = ck["counts"], ck["oris"], ck["tc"], list(ck["uids"])

In [ ]:
def signal_correlation_matrix(tc):
    """Pearson r of tuning curves between all unit pairs."""
    return np.corrcoef(tc)

r_signal = signal_correlation_matrix(tc)
print(r_signal.shape)

In [ ]:
# EXERCISE: compute the noise-correlation matrix
# YOUR CODE HERE
raise NotImplementedError('compute the noise-correlation matrix')


In [ ]:
# EXERCISE: make the signal-vs-noise scatter
# YOUR CODE HERE
raise NotImplementedError('make the signal-vs-noise scatter')


In [ ]:
# EXERCISE: (challenge) signal-vs-noise scatter, restricted to within-area pairs in V1
# YOUR CODE HERE
raise NotImplementedError('(challenge) signal-vs-noise scatter, restricted to within-area pairs in V1')


In [ ]:
save_checkpoint("module_1B", r_signal=r_signal, r_noise=r_noise, uids=np.asarray(uids))
print("✅ 1B checkpoint saved.")

## Module 1C — Functional networks & graph theory

We treat each unit as a node and connect pairs whose noise correlation exceeds a threshold.
Then we compute degree, clustering coefficient, modularity, and (challenge) communities.

In [ ]:
ck = load_or_compute("module_1B", fn=lambda: {"error": np.array([1])})
r_noise = ck["r_noise"]; uids = list(ck["uids"])
units_meta = session["units"].loc[uids]

In [ ]:
# EXERCISE: build the adjacency graph from r_noise
# YOUR CODE HERE
raise NotImplementedError('build the adjacency graph from r_noise')


In [ ]:
# EXERCISE: compute degree + clustering, plot histograms
# YOUR CODE HERE
raise NotImplementedError('compute degree + clustering, plot histograms')


In [ ]:
hub_idx = deg.argsort()[-5:][::-1]
print("Top 5 hubs:")
for i in hub_idx:
    print(f"  unit {uids[i]} ({units_meta.iloc[i]['ecephys_structure_acronym']}): degree={deg[i]}")

# Density per area
for area, ar in units_meta.groupby("ecephys_structure_acronym"):
    idx = [uids.index(u) for u in ar.index]
    sub = G.subgraph(idx)
    print(f"{area}: density={nx.density(sub):.3f} ({sub.number_of_nodes()} units)")

In [ ]:
# EXERCISE: (challenge) Louvain community detection + overlay vs anatomical area
# YOUR CODE HERE
raise NotImplementedError('(challenge) Louvain community detection + overlay vs anatomical area')


In [ ]:
plotting.plot_network(G)
plt.title("Functional network (noise corr > 0.1)");